In [ ]:
import json
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, confusion_matrix, classification_report

LOGS_DIR = Path('../logs')

assert LOGS_DIR.exists(), f"Logs directory not found: {LOGS_DIR.resolve()}"

In [ ]:
def load_all_runs(logs_dir: Path):
    records = []
    runs_raw = {}
    metrics_raw = {}

    for result_path in sorted(logs_dir.glob('*/test_results.json')):
        with open(result_path) as f:
            data = json.load(f)

        run_id = data['run_id']
        runs_raw[run_id] = data

        # infer dataset name from csv_path
        csv_stem = Path(data['csv_path']).stem 
        dataset_map = {
            'processed_landmarks': 'landmarks',
            'featured_angles': 'angles',
            'featured_distances': 'distances',
            'featured_combined': 'combined',
        }
        dataset = dataset_map.get(csv_stem, csv_stem)

        f1_macro = f1_score(data['labels'], data['predictions'], average='macro')
        f1_weighted = f1_score(data['labels'], data['predictions'], average='weighted')

        records.append({
            'run_id': run_id,
            'dataset': dataset,
            'model': data['model_name'],
            'input_size': data['input_size'],
            'test_accuracy': data['test_accuracy'],
            'test_loss': data['test_loss'],
            'best_val_loss': data['best_val_loss'],
            'f1_macro': round(f1_macro, 6),
            'f1_weighted': round(f1_weighted, 6),
            'stopped_early': data['stopped_early'],
        })

        # load metrics csv if present
        metrics_path = result_path.parent / 'metrics.csv'
        if metrics_path.exists():
            metrics_raw[run_id] = pd.read_csv(metrics_path)

    df = pd.DataFrame(records)
    return df, runs_raw, metrics_raw


df, runs_raw, metrics_raw = load_all_runs(LOGS_DIR)
print(f"{len(df)} runs loaded")
df.sort_values('test_accuracy', ascending=False).reset_index(drop=True)

In [ ]:
# --- ranking table ---
cols = ['dataset', 'model', 'test_accuracy', 'f1_macro', 'test_loss', 'best_val_loss', 'stopped_early', 'input_size']
display(df[cols].sort_values('test_accuracy', ascending=False).reset_index(drop=True))

In [ ]:
# --- heatmap: accuracy ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric, title in zip(
    axes,
    ['test_accuracy', 'f1_macro'],
    ['Test accuracy', 'F1 macro']
):
    pivot = df.pivot_table(index='dataset', columns='model', values=metric)
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.4f',
        cmap='YlGn',
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Model')
    ax.set_ylabel('Dataset')

plt.tight_layout()
plt.show()

In [ ]:
def plot_training_curves(metrics_raw: dict, runs_df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for run_id, metrics_df in metrics_raw.items():
        run_info = runs_df[runs_df['run_id'] == run_id].iloc[0]
        label = f"{run_info['dataset']} / {run_info['model']}"

        axes[0].plot(metrics_df['epoch'], metrics_df['val_loss'], label=label)
        axes[1].plot(metrics_df['epoch'], metrics_df['val_accuracy'], label=label)

        # mark early stopping
        if run_info['stopped_early']:
            last_epoch = metrics_df['epoch'].max()
            axes[0].axvline(last_epoch, linestyle='--', alpha=0.4)
            axes[1].axvline(last_epoch, linestyle='--', alpha=0.4)

    axes[0].set_title('Validation loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('Validation accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1)
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_training_curves(metrics_raw, df)

In [ ]:
def plot_training_curves_individual(metrics_raw: dict, runs_df: pd.DataFrame, n_cols: int = 3):
    run_ids = list(metrics_raw.keys())
    n_runs = len(run_ids)
    n_rows = math.ceil(n_runs / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).flatten()

    for i, run_id in enumerate(run_ids):
        metrics_df = metrics_raw[run_id]
        run_info = runs_df[runs_df['run_id'] == run_id].iloc[0]
        label = f"{run_info['dataset']} / {run_info['model']}"

        ax_loss = axes[i]
        ax_acc = ax_loss.twinx()

        # validação (curva principal)
        ax_loss.plot(metrics_df['epoch'], metrics_df['val_loss'], color='tab:red', label='val_loss')
        ax_acc.plot(metrics_df['epoch'], metrics_df['val_accuracy'], color='tab:blue', label='val_accuracy')

        # treino (tracejado, mais claro)
        if 'train_loss' in metrics_df.columns:
            ax_loss.plot(metrics_df['epoch'], metrics_df['train_loss'],
                         color='tab:red', linestyle='--', alpha=0.4, label='train_loss')
        if 'train_accuracy' in metrics_df.columns:
            ax_acc.plot(metrics_df['epoch'], metrics_df['train_accuracy'],
                        color='tab:blue', linestyle='--', alpha=0.4, label='train_accuracy')

        if run_info['stopped_early']:
            last_epoch = metrics_df['epoch'].max()
            ax_loss.axvline(last_epoch, linestyle='--', alpha=0.4, color='gray')

        ax_loss.set_title(label, fontsize=10)
        ax_loss.set_xlabel('Epoch')
        ax_loss.set_ylabel('Loss', color='tab:red')
        ax_acc.set_ylabel('Accuracy', color='tab:blue')
        ax_acc.set_ylim(0, 1)
        ax_loss.grid(True, alpha=0.3)

        lines1, labels1 = ax_loss.get_legend_handles_labels()
        lines2, labels2 = ax_acc.get_legend_handles_labels()
        ax_loss.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper right')

    for j in range(n_runs, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()


plot_training_curves_individual(metrics_raw, df)

In [ ]:
def plot_confusion_matrix(run_id: str, runs_raw: dict, ax=None):
    data = runs_raw[run_id]
    labels = data['labels']
    preds = data['predictions']
    class_names = [data['class_names'][str(i)] for i in range(data['num_classes'])]

    cm = confusion_matrix(labels, preds, normalize='true')

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='.2f',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(run_id, fontsize=9)


# plot one confusion matrix per run
run_ids = list(runs_raw.keys())
n = len(run_ids)
ncols = min(n, 2)
nrows = (n + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 6 * nrows))
axes = np.array(axes).flatten()

for i, run_id in enumerate(run_ids):
    plot_confusion_matrix(run_id, runs_raw, ax=axes[i])

# hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# classification report for each run
for run_id, data in runs_raw.items():
    class_names = [data['class_names'][str(i)] for i in range(data['num_classes'])]
    print(f"\n{'='*60}")
    print(f"Run: {run_id}")
    print('='*60)
    print(classification_report(data['labels'], data['predictions'], target_names=class_names))